# Backblaze Drive Observations — Q1 2018 Bronze Ingestion

**Purpose:**  Ingest the original daily Backblaze Drive Stats CSV files for Q1 2018
into a Bronze Delta table.

**Source:**  
Unity Catalog Volume:
`/Volumes/backblaze_data/ing_bronze/source_files/extracted/2018_Q1/`

**Target:**  `backblaze_data.ing_bronze.drive_observations_2018_q1`

## Processing conventions
- Preserve source column names and values.
- Read source values as strings to preserve raw representations.
- Add ingestion metadata for lineage.
- Perform ingestion-level validation only.
- Refresh the Bronze table as a full snapshot.
- Defer date normalization, deduplication, and reliability calculations to Silver/Gold.

In [0]:
from pyspark.sql import functions as F

SOURCE_DIR = (
    "/Volumes/backblaze_data/ing_bronze/source_files/"
    "extracted/2018_Q1"
)

TARGET_TABLE = (
    "backblaze_data.ing_bronze.drive_observations_2018_q1"
)

EXPECTED_SOURCE_COLUMNS = 105

print(f"Source directory: {SOURCE_DIR}")
print(f"Target table: {TARGET_TABLE}")

In [0]:
source_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("mode", "FAILFAST")
    .csv(f"{SOURCE_DIR}/*.csv")
)

source_columns = source_df.columns

print(f"Source columns: {len(source_columns)}")
print(f"Source rows: {source_df.count()}")
print("First 20 source columns:")
print(source_columns[:20])

source_df.printSchema()

In [0]:
required_columns = {
    "date",
    "serial_number",
    "model",
    "capacity_bytes",
    "failure",
}

missing_columns = required_columns - set(source_columns)

if len(source_columns) != EXPECTED_SOURCE_COLUMNS:
    raise ValueError(
        f"Expected {EXPECTED_SOURCE_COLUMNS} source columns, "
        f"found {len(source_columns)}"
    )

if missing_columns:
    raise ValueError(
        f"Missing required source columns: {sorted(missing_columns)}"
    )

bronze_df = (
    source_df
    .withColumn(
        "_source_file_path",
        F.col("_metadata.file_path")
    )
    .withColumn(
        "_ingested_at",
        F.current_timestamp()
    )
)

print("Source schema validation passed.")
print(f"Source columns: {len(source_columns)}")
print(f"Columns including metadata: {len(bronze_df.columns)}")

In [0]:
(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print(f"Bronze snapshot written successfully: {TARGET_TABLE}")

In [0]:
bronze_table_df = spark.table(TARGET_TABLE)

bronze_count = bronze_table_df.count()

print(f"Target table: {TARGET_TABLE}")
print(f"Rows written: {bronze_count}")
print(f"Columns in Bronze table: {len(bronze_table_df.columns)}")

display(
    bronze_table_df
    .select(
        "date",
        "serial_number",
        "model",
        "capacity_bytes",
        "failure",
        "_source_file_path",
        "_ingested_at",
    )
    .limit(10)
)

In [0]:
from pyspark.sql import functions as F

validation = (
    spark.table(TARGET_TABLE)
    .agg(
        F.count("*").alias("total_rows"),
        F.countDistinct("_source_file_path").alias("source_files"),
        F.sum(
            F.when(F.col("_source_file_path").isNull(), 1).otherwise(0)
        ).alias("null_source_paths"),
        F.sum(
            F.when(F.col("_ingested_at").isNull(), 1).otherwise(0)
        ).alias("null_ingestion_timestamps"),
    )
)

display(validation)